# Matched-information sequence models

Executable scientific definitions and computed results are presented below. Data, fitted models, tables and figures are stored in the corresponding standard project directories. Earlier experiments are preserved separately in `Data/legacy/Notebooks/` and are not mixed with the current results.

In [1]:
from pathlib import Path
import sys, types, hashlib, importlib.abc, importlib.util
import nbformat
import pandas as pd
from IPython.core.magic import register_cell_magic
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Notebooks').is_dir() and (p/'Data').is_dir())
MODULE_NOTEBOOKS={'revision_data': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_models': '03_baseline_models.ipynb', 'revision_sensitivity': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_neural': '04_sota_models.ipynb', 'revision_evaluation': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_supplemental': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_provenance': '05_proposed_hybrid_model_and_ablations.ipynb', 'revision_validation': '02_data_preprocessing_feature_engineering_and_splits.ipynb', 'revision_closure': '06_final_validation_tables_figures_and_reports.ipynb', 'revision_status': '06_final_validation_tables_figures_and_reports.ipynb'}

class NotebookSourceLoader(importlib.abc.Loader):
    def create_module(self,spec):return None
    def exec_module(self,module):
        path=ROOT/'Notebooks'/MODULE_NOTEBOOKS[module.__name__]
        notebook=nbformat.read(path,4)
        cell=next(c for c in notebook.cells if c.metadata.get('research_module')==module.__name__)
        source=cell.source.split('\n',1)[1]
        module.__file__=str(path)
        module.__notebook_source_sha256__=hashlib.sha256(source.encode()).hexdigest()
        exec(compile(source,str(path)+'#'+cell.id,'exec'),module.__dict__)

class NotebookSourceFinder(importlib.abc.MetaPathFinder):
    def find_spec(self,fullname,path=None,target=None):
        if fullname in MODULE_NOTEBOOKS:
            return importlib.util.spec_from_loader(fullname,NotebookSourceLoader())
sys.meta_path=[f for f in sys.meta_path if type(f).__name__!='NotebookSourceFinder']
sys.meta_path.insert(0,NotebookSourceFinder())

@register_cell_magic
def research_module(line,source):
    """Publish the visible functions for reuse by other notebooks; no hidden helper scripts."""
    name=line.strip();digest=hashlib.sha256(source.rstrip('\n').encode()).hexdigest()
    existing=sys.modules.get(name)
    if existing is not None and existing.__notebook_source_sha256__==digest:return
    module=types.ModuleType(name);module.__file__=str(ROOT/'Notebooks'/MODULE_NOTEBOOKS[name])
    module.__notebook_source_sha256__=digest;sys.modules[name]=module
    exec(compile(source.rstrip('\n'),module.__file__,'exec'),module.__dict__)

@register_cell_magic
def legacy_snapshot(line,cell):
    """Archived analysis is preserved but is not part of the current execution."""
    return None

import revision_data as rd
artifact_path=rd.artifact_path
import matplotlib as mpl
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('png')
mpl.rcParams.update({'figure.dpi':350,'savefig.dpi':350})
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',100)

### Neural — executable definitions

The functions below are the source used by this notebook and reused by the other notebooks. Defining them does not repeat model fitting.

In [2]:
%%research_module revision_neural
"""Prespecified GRU/SAKT proxies and HGB with identical sequence information."""
from revision_data import artifact_path, code_digest, source_matches, artifact_matches, fit_contract_matches
import ast
import copy
import hashlib
import io
import math
import random
import time
import nbformat
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader,Subset
from threadpoolctl import threadpool_limits
from revision_data import ROOT,V,KEYS,protocol,digest,save_json,table
from revision_models import make_hgb,metrics,fit_calibrator,calibrate,prediction_frame

def matched_sequence_experiment():
    pr=protocol();cfg=pr['neural'];d=pd.read_parquet(artifact_path('Data/features_primary.parquet')).sort_values(['DateAnswered','AnswerId']).reset_index(drop=True)
    fit=np.flatnonzero(d.split.eq('fit'));dev=np.flatnonzero(d.split.eq('development'));cal=np.flatnonzero(d.split.eq('calibration'));ev=np.flatnonzero(d.split.eq('evaluation'))
    d[KEYS+['DateAnswered','split']].to_parquet(artifact_path('Data/neural_exact_split_ids.parquet'),index=False)
    qmap={q:i+1 for i,q in enumerate(sorted(d.iloc[fit].QuestionId.unique()))};q=d.QuestionId.map(qmap).fillna(0).to_numpy('int64');nq=len(qmap)+1;L=cfg['history_length'];N=len(d)
    hq=np.zeros((N,L),dtype='int64');hr=np.zeros((N,L),dtype='int64');mask=np.zeros((N,L),bool)
    history={}
    for _,g in d.groupby('DateAnswered',sort=False):
        for i,r in zip(g.index,g.itertuples(index=False)):
            batches=history.get(r.UserId,[]);chosen=[];count=0
            for batch in reversed(batches):
                if count+len(batch)>L:break
                chosen.append(batch);count+=len(batch)
            past=[j for batch in reversed(chosen) for j in batch]
            if past:hq[i,-len(past):]=q[past];hr[i,-len(past):]=d.IsCorrect.iloc[past];mask[i,-len(past):]=True
        for u,part in g.groupby('UserId'):history.setdefault(u,[]).append(part.index.tolist())
    numeric=np.column_stack([q,hq,hr,mask.astype(int)]).astype('float32')
    dataset=TensorDataset(torch.zeros((N,1)),torch.from_numpy(hq),torch.zeros((N,L),dtype=torch.long),torch.from_numpy(hr),torch.zeros((N,L)),torch.from_numpy(mask),torch.from_numpy(q),torch.zeros(N,dtype=torch.long),torch.zeros(N,dtype=torch.long),torch.from_numpy(d.IsCorrect.to_numpy('float32')))
    original=nbformat.read(next((artifact_path('legacy/Notebooks')).glob('04_*.ipynb')),as_version=4)
    source=next(c.source for c in original.cells if c.id=='cf6ef359');nodes=[x for x in ast.parse(source).body if isinstance(x,ast.ClassDef) and x.name in ['DKT','SAKT']]
    env={'torch':torch,'nn':nn,'math':math,'NQ':nq,'NS':1,'L':L,'D':24}
    exec(compile(ast.Module(body=nodes,type_ignores=[]),'<existing GRU and SAKT>','exec'),env)
    device=torch.device('mps' if torch.backends.mps.is_available() else 'cpu');torch.set_num_threads(4)
    curves=[];registry=[];predictions=[]
    def loader(ids,shuffle=False):return DataLoader(Subset(dataset,ids.tolist()),batch_size=cfg['batch_size'],shuffle=shuffle)
    def forward(model,ids):
        model.eval();result=[]
        with torch.no_grad():
            for batch in loader(ids):result.append(torch.sigmoid(model(*[t.to(device) for t in batch[:-1]])).cpu().numpy())
        return np.concatenate(result)
    lossfn=nn.BCEWithLogitsLoss()
    for seed in cfg['seeds']:
        for name,classname in [('DKT-inspired GRU proxy','DKT'),('SAKT-inspired attention proxy','SAKT')]:
            candidates=[]
            for rate in cfg['learning_rates']:
                random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
                model=env[classname]();initial=io.BytesIO();torch.save(model.state_dict(),initial);initial_hash=hashlib.sha256(initial.getvalue()).hexdigest();model=model.to(device)
                opt=torch.optim.AdamW(model.parameters(),lr=rate,weight_decay=1e-4);best_loss=np.inf;best=None;stale=0;reason='budget_limited';start=time.perf_counter()
                for epoch in range(1,cfg['max_epochs']+1):
                    model.train();loss_sum=0.;count=0
                    for batch in loader(fit,True):
                        tensors=[t.to(device) for t in batch];opt.zero_grad(set_to_none=True);loss=lossfn(model(*tensors[:-1]),tensors[-1]);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1);opt.step();loss_sum+=float(loss.detach().cpu())*len(batch[-1]);count+=len(batch[-1])
                    p=forward(model,dev);vl=float(-np.mean(d.IsCorrect.iloc[dev]*np.log(np.clip(p,1e-6,1))+(1-d.IsCorrect.iloc[dev])*np.log(np.clip(1-p,1e-6,1))))
                    curves.append({'model':name,'seed':seed,'learning_rate':rate,'epoch':epoch,'train_loss':loss_sum/count,'development_loss':vl,'initialization_hash':initial_hash,'stage':'matched_information'})
                    pd.DataFrame(curves).to_parquet(artifact_path('Tables/learning_curves.parquet'),index=False)
                    if vl<best_loss-1e-4:best_loss=vl;best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};stale=0
                    else:stale+=1
                    print(f'{name} seed={seed} lr={rate} epoch={epoch} development_loss={vl:.5f}',flush=True)
                    if stale>=cfg['patience']:reason='early_stopping_patience';break
                model.load_state_dict(best);candidates.append((best_loss,rate,best,reason,epoch,initial_hash))
                registry.append({'model':name,'seed':seed,'learning_rate':rate,'epochs':epoch,'stop_reason':reason,'initialization_hash':initial_hash,'development_loss':best_loss,'seconds':time.perf_counter()-start,'fit_rows':len(fit),'target_rows':len(ev),'history_length':L,'canonical_SOTA':False})
            loss,rate,weights,reason,epochs,initial_hash=min(candidates,key=lambda x:x[0]);model.load_state_dict(weights);model=model.to(device)
            pc=forward(model,cal);pe=forward(model,ev);calibrator=fit_calibrator('platt',pc,d.IsCorrect.iloc[cal]);p=calibrate(calibrator,pe)
            path=artifact_path('Models')/f'matched_{classname}_{seed}.pt';torch.save({'state_dict':weights,'question_map':qmap,'history_length':L,'seed':seed,'learning_rate':rate,'initialization_hash':initial_hash,'calibrator':calibrator,'information':'current question + prior 30 q/response events, whole tie batches'},path)
            predictions.append(prediction_frame(d.iloc[ev],p,name,seed,0,'frozen_object','matched_information_30',digest(path)))
        for trees in [80,100]:
            options=[]
            with threadpool_limits(limits=4):
                for l2 in pr['hgb']['l2_candidates']:
                    m=make_hgb(trees,l2,seed);m.fit(numeric[fit],d.IsCorrect.iloc[fit]);p=m.predict_proba(numeric[dev])[:,1];ll=float(-np.mean(d.IsCorrect.iloc[dev]*np.log(np.clip(p,1e-6,1))+(1-d.IsCorrect.iloc[dev])*np.log(np.clip(1-p,1e-6,1))));options.append((ll,l2,m))
                    registry.append({'model':f'HGB{trees}','seed':seed,'l2':l2,'development_loss':ll,'fit_rows':len(fit),'target_rows':len(ev),'history_length':L,'stop_reason':'fixed_tree_budget','question_encoding':'same categorical identities encoded as numeric indices; not ordered knowledge'})
            ll,l2,m=min(options,key=lambda x:x[0]);pc=m.predict_proba(numeric[cal])[:,1];pe=m.predict_proba(numeric[ev])[:,1];calibrator=fit_calibrator('platt',pc,d.IsCorrect.iloc[cal]);p=calibrate(calibrator,pe)
            import joblib
            path=artifact_path('Models')/f'matched_HGB{trees}_{seed}.joblib';joblib.dump({'model':m,'calibrator':calibrator,'qmap':qmap,'features':'question ID, all 30 prior question IDs, responses, masks'},path)
            predictions.append(prediction_frame(d.iloc[ev],p,f'HGB{trees}',seed,0,'frozen_object','matched_information_30',digest(path)))
        pd.DataFrame(registry).to_csv(artifact_path('Tables/candidate_registry.csv'),index=False)
        pd.concat(predictions,ignore_index=True).to_parquet(artifact_path('Data/matched_information_predictions.parquet'),index=False)
    rows=[]
    for g in predictions:
        rows.append({'model':g.model.iloc[0],'seed':g.seed.iloc[0],'N':len(g),**metrics(g.y,g.p)})
    return table('matched_information_results.csv',pd.DataFrame(rows),'NB04')

In [2]:
import revision_neural as rn
display(rn.matched_sequence_experiment())

DKT-inspired GRU proxy seed=20260713 lr=0.001 epoch=1 development_loss=0.65507


DKT-inspired GRU proxy seed=20260713 lr=0.001 epoch=2 development_loss=0.63985


DKT-inspired GRU proxy seed=20260713 lr=0.001 epoch=3 development_loss=0.64159


DKT-inspired GRU proxy seed=20260713 lr=0.001 epoch=4 development_loss=0.64382


DKT-inspired GRU proxy seed=20260713 lr=0.001 epoch=5 development_loss=0.65037


...

SAKT-inspired attention proxy seed=20260715 lr=0.0003 epoch=5 development_loss=0.65286


SAKT-inspired attention proxy seed=20260715 lr=0.0003 epoch=6 development_loss=0.65370


SAKT-inspired attention proxy seed=20260715 lr=0.0003 epoch=7 development_loss=0.65524


SAKT-inspired attention proxy seed=20260715 lr=0.0003 epoch=8 development_loss=0.65607


,model,seed,N,ROC_AUC,AP_correct,AP_incorrect,Log_Loss,Brier,ECE15,Calibration_Intercept,Calibration_Slope,Accuracy,NA_reason
0,DKT-inspired GRU proxy,20260713,143565,0.609506,0.726087,0.449040,0.633731,0.221338,0.021707,-0.258788,1.232518,0.647839,
1,SAKT-inspired attention proxy,20260713,143565,0.639556,0.746493,0.471570,0.625784,0.217891,0.033145,-0.382593,1.347487,0.647337,
2,HGB80,20260713,143565,0.730068,0.829854,0.567933,0.571078,0.195066,0.010206,-0.040845,0.994513,0.697802,
3,HGB100,20260713,143565,0.729958,0.829599,0.567699,0.571178,0.195107,0.010255,-0.041088,0.995553,0.697983,
4,DKT-inspired GRU proxy,20260714,143565,0.622286,0.736298,0.459431,0.630106,0.219710,0.028299,-0.220790,1.125602,0.650242,
5,SAKT-inspired attention proxy,20260714,143565,0.625819,0.739543,0.456560,0.630618,0.220017,0.033006,-0.333678,1.270295,0.647484,
6,HGB80,20260714,143565,0.730068,0.829854,0.567933,0.571078,0.195066,0.010206,-0.040845,0.994513,0.697802,
7,HGB100,20260714,143565,0.729958,0.829599,0.567699,0.571178,0.195107,0.010255,-0.041088,0.995553,0.697983,
8,DKT-inspired GRU proxy,20260715,143565,0.632315,0.745373,0.469956,0.626378,0.218045,0.028566,-0.184146,1.071823,0.652011,
9,SAKT-inspired attention proxy,20260715,143565,0.617455,0.729204,0.451973,0.634278,0.221644,0.033039,-0.431936,1.391656,0.647484,
